In [ ]:
# Cài đặt
!pip -q uninstall -y hf-xet
!pip -q install -U fastapi uvicorn sentence-transformers pyngrok huggingface_hub

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from pyngrok import ngrok
import uvicorn
from threading import Thread

# Khởi tạo
app = FastAPI(title="CV-JD BGE-M3 Embedding API")
print("Đang tải BAAI/bge-m3...")

model = SentenceTransformer(
    "BAAI/bge-m3",
    trust_remote_code=True,
    revision="main"
)

print("Tải model thành công!")

# API
class TextRequest(BaseModel):
    text: str

@app.post("/api/embed")
def embed(req: TextRequest):
    return {"embedding": model.encode(req.text).tolist()}

# Ngrok
ngrok.set_auth_token("==========SECRET KEY==========")
url = ngrok.connect(8000).public_url

print("=" * 50)
print(f"API URL: {url}/api/embed")
print("=" * 50)

# Chạy server
Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
).start()

print("FastAPI BGE-M3 đang chạy!")